<a href="https://colab.research.google.com/github/iws3/llm_from_Scratch/blob/main/llm_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from datasets import load_dataset
from tqdm.auto import tqdm

NUM_SAMPLES = 100_000  # Start with 100K samples (~50-100MB text)

print("Loading CulturaX Urdu dataset (streaming)...")
dataset = load_dataset(
    "uonlp/CulturaX",
    "ur",                    # Urdu language code
    split="train",
    streaming=True          # Don't download everything
)

# Collect samples
raw_texts = []
for i, sample in enumerate(tqdm(dataset, total=NUM_SAMPLES, desc="Downloading")):
    if i >= NUM_SAMPLES:
        break
    raw_texts.append(sample["text"])

print(f"\nDownloaded {len(raw_texts)} samples")
print(f"Total characters: {sum(len(t) for t in raw_texts):,}")
print(f"\nSample text (first 500 chars):")
print(raw_texts[0][:500])

Loading CulturaX Urdu dataset (streaming)...


Downloading:   0%|          | 0/100000 [00:00<?, ?it/s]


Downloaded 100000 samples
Total characters: 365,034,260

Sample text (first 500 chars):
موجودہ حکومت کے غریب عوام کے لیے اقدامات۔۔چوہدری محمد ایوب - مکالمہمکالمہ
ایک نجی بیٹھک میں دوستوں کے ساتھ گپ شپ ہورہی تھی اور مختلف ادوار میں مختلف حکومتوں کی طرف سے عوام کو ریلیف دینے کے لیے شروع کیے گئے پروگرامز، لیپ ٹاپ، سستی روٹی،بینظیر انکم سپورٹ پروگرام یا اب موجودہ گورنمنٹ کی طر ف سے شیلٹر ہومز اور لنگر خانے وغیرہ، یہ وہ چیزیں ہیں جن سے عام آدمی کو تو شاید فائدہ نہیں پہنچا بلکہ ان میں کہیں نہ کہیں کوئی نہ کوئی کرپشن ضرور ہو جاتی ہے اور بعد میں آنے والی حکومتیں پچھلی حکومتوں پر الزام لگات


In [13]:
import re
import unicodedata
import regex

def clean_urdu_text(text: str) -> str:
    """
    Clean a single Urdu text document.

    Steps:
    1. Remove URLs
    2. Remove HTML tags and entities
    3. Remove email addresses
    4. Normalize Unicode (NFKC normalization)
    5. Remove non-Urdu characters (keep Urdu + punctuation + digits)
    6. Normalize repeated punctuation (۔۔۔, ..., - -, etc.)
    7. Normalize whitespace
    """

    # Step 1: Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Step 2: Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove HTML entities
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)

    # Step 3: Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # Step 4: Unicode normalization (NFKC)
    # This normalizes different representations of the same character
    text = unicodedata.normalize('NFKC', text)

    # Step 5: Keep only Urdu characters, basic punctuation, digits, and whitespace
    # Urdu Unicode ranges + Arabic punctuation + Western digits + basic punctuation
    urdu_pattern = regex.compile(
        r'[^'
        r'\u0600-\u06FF'    # Arabic (includes Urdu)
        r'\u0750-\u077F'    # Arabic Supplement
        r'\u08A0-\u08FF'    # Arabic Extended-A
        r'\uFB50-\uFDFF'    # Arabic Presentation Forms-A
        r'\uFE70-\uFEFF'    # Arabic Presentation Forms-B
        r'0-9۰-۹'           # Western and Eastern Arabic-Indic digits
        r'\s'               # Whitespace
        r'۔،؟!٪'           # Urdu punctuation (full stop, comma, question mark, etc.)
        r'.,:;!?\-\("\']'  # Basic Latin punctuation
    )
    text = urdu_pattern.sub(' ', text)

    # Step 6: Normalize repeated punctuation
    text = re.sub(r'۔{2,}', '۔', text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'-\s*-+', '-', text)
    text = re.sub(r'-{2,}', '-', text)
    text = re.sub(r'،{2,}', '،', text)
    text = re.sub(r',{2,}', ',', text)
    text = re.sub(r'\s+[۔\.\-,،]\s+', ' ', text)

    # Step 7: Normalize whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)  # Max 2 newlines
    text = re.sub(r'[^\S\n]+', ' ', text)    # Collapse spaces (but keep newlines)
    text = text.strip()

    return text


def is_mostly_urdu(text: str, threshold: float = 0.5) -> bool:
    """
    Check if text is mostly Urdu characters.
    This filters out documents that are primarily English/other languages.

    threshold: minimum fraction of characters that must be Urdu
    """
    if len(text) == 0:
        return False
    urdu_chars = len(regex.findall(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]', text))
    return (urdu_chars / len(text)) > threshold


# Test the cleaning function
sample = raw_texts[0]
print("=== BEFORE CLEANING ===")
print(sample[:300])
print("\n=== AFTER CLEANING ===")
cleaned = clean_urdu_text(sample)
print(cleaned[:300])
print(f"\nIs mostly Urdu: {is_mostly_urdu(cleaned)}")

=== BEFORE CLEANING ===
موجودہ حکومت کے غریب عوام کے لیے اقدامات۔۔چوہدری محمد ایوب - مکالمہمکالمہ
ایک نجی بیٹھک میں دوستوں کے ساتھ گپ شپ ہورہی تھی اور مختلف ادوار میں مختلف حکومتوں کی طرف سے عوام کو ریلیف دینے کے لیے شروع کیے گئے پروگرامز، لیپ ٹاپ، سستی روٹی،بینظیر انکم سپورٹ پروگرام یا اب موجودہ گورنمنٹ کی طر ف سے شیلٹر ہ

=== AFTER CLEANING ===
موجودہ حکومت کے غریب عوام کے لیے اقدامات۔چوہدری محمد ایوب مکالمہمکالمہ
ایک نجی بیٹھک میں دوستوں کے ساتھ گپ شپ ہورہی تھی اور مختلف ادوار میں مختلف حکومتوں کی طرف سے عوام کو ریلیف دینے کے لیے شروع کیے گئے پروگرامز، لیپ ٹاپ، سستی روٹی،بینظیر انکم سپورٹ پروگرام یا اب موجودہ گورنمنٹ کی طر ف سے شیلٹر ہومز

Is mostly Urdu: True
